In [ ]:
import os
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt

#### Input

In [ ]:
basedir = '/Volumes/External/ISMIP7/AIS/'
model = 'MRI-ESM2-0'
domain = 'GEMB-SDBN1-8000m'
version = 'v2'
scenario = 'ssp585'

sec_per_year = 31556943.36
freshwater_density = 1000.0

#### SMB anomaly

In [ ]:
#Historical
path = os.path.join(basedir,model,'historical',domain,'acabf-anomaly',version,'*')
ds = xr.open_mfdataset(path)
ds = ds.sel(time=slice('1995-01-01', '2025-01-01'))

#Future bit
path = os.path.join(basedir,model,scenario,domain,'acabf-anomaly',version,'*20*.nc')
ds2 = xr.open_mfdataset(path)
ds2 = ds2.sel(time=slice('1995-01-01', '2025-01-01'))

# Paste the two together
smb = xr.concat([ds['acabf-anomaly'], ds2['acabf-anomaly']], dim='time')

# Take a time-mean
weights = smb.time.dt.daysinmonth
smb_mean = (smb * weights).sum(dim='time') / weights.sum()

# Convert to m.w.e. / yr
smb_mean = (smb_mean * sec_per_year / freshwater_density).rename('SMB')

# Save to netcdf
smb_mean.to_netcdf(f'/Volumes/External/ISMIP7/AIS/{model}/SMB_offset_{model}_{scenario}.nc')